In [3]:
import numpy as np

# Primary Task

Firstly simulate patients arriving in a year. 

In [4]:
# We assume that the arrival rates are constant for each day and so we createa poisson process for each day. 
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    t =1
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []
    while t <= 365: 
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients


In [5]:
def lam1(t): 
   return -(1/3650)*t**2 + (1/10)*t

def lam2(t): 
   return lam1(t)/5

def lam3(t):
   return 6


X = arrivals_year(lam1,lam2,lam3)

In [6]:
len(X)

4790

In [27]:
# System of wards
def system(bedsA,bedsB,bedsC, patientflow_year):
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []


    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                blocked_B += 1
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


## Performance measures

In [85]:
# Crude monte carlo estimator
def probs(bedsA,bedsB,bedsC,n):
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lam1,lam2,lam3)
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)

        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(a/type_A)
        frac_B.append(b/type_B)
        frac_C.append(c/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

    

In [9]:
np.random.seed(42)
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(15,15,45,1000)

In [10]:
meanbedA

0.9377517270681892

In [11]:
meanbedB

0.7465595645542884

In [12]:
meanbedC

0.9309626991922798

In [13]:
meanall

2242.0

In [14]:
np.random.seed(42)
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(15,15,45,100)

In [15]:
meanall

2149.0

# Sensitivity analysis

In [80]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
# This is monte carlo
def sum_relocated(bedsA,bedsB,bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C.patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across wards A, B and C
    A = []
    B = []
    C = []

    for X in patient_flows:
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(all), np.var(all)

    

In [81]:
def system_control(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and LOS_i is the mean value of LOS in use in ward i
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)

            else:
                blocked_B += 1
                LOS_B.append(LOS)
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    blocked_A+=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)


            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A),np.mean(LOS_B), np.mean(LOS_C)

# Control variate function
def control_variate_sum(bedsA, bedsB, bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across ward A, B and C. 
    #### Find ci
    # Initialize
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[:50]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find ci
    ca = -np.cov(A,LA)[0,1]/np.var(LA)
    cB = -np.cov(B,LB)[0,1]/np.var(LB)
    cC = -np.cov(C,LC)[0,1]/np.var(LC)

    # Reinitialize to actually find the control variates
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[50:]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find new variable
    Ya = A+ca*(LA-8) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanA = np.mean(Ya)

    YB = B+cB*(LB-12) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanB = np.mean(YB)

    YC = C+cC*(LC-10) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanC = np.mean(YC)



    return meanA+meanB+meanC, np.var(Ya)+np.var(YB)+np.var(YC)+2*np.cov(Ya,YB)[0,1]+2*np.cov(YB,YC)[0,1]+2*np.cov(Ya,YC)[0,1]

# Now doing sensitivity

In [86]:
# Optimal found by Katrinw 22,14,39

fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(22,14,39,1000)

In [87]:
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC 

(0.6360614007579427,
 0.24750825617220645,
 0.30587182627332843,
 1412.042,
 110.476,
 670.222,
 2192.74,
 0.9185357954880676,
 0.7620212515884798,
 0.9460332367483961)

## Exponentiel

In [92]:
X = arrivals_year(lam1,lam2,lam3)
system_control(15,15,45,X)

(1680, 92, 467, 7.069367590410722, 10.290211628111253, 9.217675304481068)